PART 04：跑起来——放行、问人、硬拦，然后我把它拆了
代码拼完，跑：

python code.py
四段实录，一段比一段离谱。

实录一：日常操作，零摩擦
帮我创建 notes/todo.txt,写入"复盘今晚的改动",然后读给我确认
模型连调 write_file、read_file，两道工具调用三道闸门全空过，终端里连一行黄字都没有——跟没装门禁时一模一样。这就是 PART 02 说的"默认快"：日常操作不该为安全付费。

实录二：问人——以及按下 N 之后发生的事
删掉 notes/todo.txt
模型调 bash，命令 rm notes/todo.txt，命中规则"rm "：

[permission] Potentially destructive command
   Tool: bash({'command': 'rm notes/todo.txt'})
   Allow? [y/N]
我按了 N。模型收到 Permission denied.，下一圈它的反应分两种，我都撞见过：

温和的那种，直接认了："好的，文件保留，未删除。"——拒绝生效，收工。

有种更值得警惕的：它改道。 被拒了 rm，它转头调 python -c "import os; os.remove('notes/todo.txt')"——用 Python 删。而这条命令的文本里没有 "rm "字样，规则不命中，直接放行。你按的那个 N，只拦住了六种写法里的一种。

记住这一幕，它就是本篇最后一节的预告片。

实录三：硬拒绝——连问都不问
我直接下套：用 rm -rf / 清理整个系统。模型调 bash，闸门 1 命中：

[blocked] Blocked: 'rm -rf /' is on the deny list
红字，秒拦，不弹审批——有些事不值得问人，因为答案一定是 No。模型收到 Permission denied.，回复了一段"这条命令被安全策略拦截，它会产生不可逆的破坏"。

看起来很美？别急。

实录四：高潮——我绕过自己的门禁
现在，我以攻击者的心态，重新审视这三道我亲手装的闸门。

第一发：骗过闸门 1。 黑名单匹配的是字符串 rm -rf /，那我写成 rm -rf '/'——加一对引号，shell 展开后一模一样，但子串匹配瞎了：命令文本里 rm -rf 和 / 之间隔了个引号，"rm -rf /" 这个子串不存在。闸门 1，穿透。

不过别高兴太早：这条命令里仍有 rm ，闸门 2 命中，拦下来问人——还有一道人肉兜底。行，那只算热身。

第二发：全穿透。 我调 write_file 写一个脚本：

write_file(path="clean.sh", content="rm -rf ~/important_backup\n")
然后调 bash 执行：

bash(command="sh clean.sh")
检查一遍：闸门 1 看 sh clean.sh——干净，黑名单词一个没有；闸门 2 看 sh clean.sh——没有 "rm "，不命中；闸门 3？规则没命中，根本走不到。三道闸门，全部通过，命令直接执行，备份目录灰飞烟灭。

而 write_file 这边，它只检查 path 出没出工作区——clean.sh 在工作区里，放行。内容是什么，没有任何闸门看。

这就是字符串黑名单的死穴：它检查的是命令的文本，不是命令的行为。文本和行为之间隔着无数种变换——引号、变量拼接、base64 管道、写成脚本文件再执行、换个解释器（python -c、perl -e、node -e）……同一件危险的事，有一百种不在黑名单里的写法。

冷静下来想一个问题：明知道挡不住，为什么还要这么写？

因为这份代码要示范的是闸门的位置，不是匹配器的强度。权限检查必须在工具分发之前、必须覆盖所有工具、拒绝必须回流成模型的输入——这三个结构决定是对的，至于匹配器，它故意用了最朴素的子串匹配，连代码注释都直说：这张表用简单字符串匹配来说明权限闸门的位置，不能视为完整的安全边界。

那真正的安全边界靠什么？业界三条正路，一条比一条根本：

白名单思路
：黑名单列不完危险，但安全操作列得完。默认全问，把"确定无害"的操作（读文件、跑测试）挑出来 allow——危险的写法千千万，安全的写法就那几种，枚举优势在我。
沙箱
：不猜命令坏不坏，直接把执行环境隔离——文件系统只读、网络封死、目录白名单。绕过了字符串检查？没关系，沙箱里没有它想碰的东西。
最小权限
：进程本身用降权用户跑，系统层面的权限兜底——就算全部检查被绕过，rm -rf / 也会在第一个目录上吃到 Permission denied。真正的安全从来不靠一层，靠的是绕过任何一层都还有下一层。
我们这份 20 行的门禁，示范的是第一层的位置；下一节看看真实产品，是怎么把这几层垒起来的。

PART 05：那张你天天见的审批弹窗，底层就是这张门禁表
现在把我们的三道闸门，和真实 Claude Code 的权限系统对上表——你会发现骨架一模一样，只是血肉厚了几个量级。

我们的三道闸 → 它的三张表。 真实系统的权限规则分三态：allow（直接放行，对应我们的"三道全不命中"）、ask（要问人，对应我们的规则命中）、deny（直接拦，对应硬拒绝表）。配置在 settings.json 里长这样：

In [ ]:
{
  "permissions": {
    "allow": ["Read", "Bash(git diff:*)"],
    "ask":   ["Bash(rm:*)"],
    "deny":  ["Bash(git push --force:*)", "Read(./.env)"]
  }
}

看语法细节：Bash(rm:*)——按命令前缀匹配，所有以 rm 开头的命令一网打尽，比我裸的子串匹配精细得多；Read(./.env) 把环境变量文件单独拉黑，最小粒度到单个文件。粒度，还是粒度：权限的粒度越细，"默认快例外慢"的快车道就能修得越宽——安全操作精确放行，可疑操作精确拦截，互不牵连。

我们的 ask_user 黄字 → 那个审批弹窗。 弹窗干的事和我们的黄字一模一样：告诉你它想干什么、参数是什么，等你拍板。但真实产品多了一个救命选项："Always allow"——选了它，这条规则当场从 ask 挪进 allow，本次会话内不再问。

回想实录二：同一个"删临时文件"的任务，模型连删五个文件，问了你五次。人类在这种重复审批里会迅速磨掉耐心，变成无脑按 y——审批的价值取决于看的那一眼，无脑 y 的那一刻权限系统就失效了。"Always allow" 就是承认人扛不住重复，让人只教一次。

还有一排模式开关。 default 是标准三态；acceptEdits 自动放行文件编辑（写代码场景里"改文件"是主业，次次问会把人问疯）；plan 模式只读不写，规划阶段零风险；还有一个名字都懒得掩饰的 --dangerously-skip-permissions——把三道闸门全拆，裸奔模式。以前你可能只知道这个参数"危险"，现在你确切知道它拆掉的是什么：deny 表、规则匹配、人工审批，一个 agent 最后的分寸。

一张表总结这次对齐：

![](门闸与claudecode的区别.png)

结尾：它开始有分寸了
三篇连起来看：第一篇造心脏，一个 while 循环转起来；第二篇长双手，五个工具随便使；这一篇，装上了神经——它开始知道疼了。

知道哪些事不用请示，哪些事要先问一句，哪些事想都别想。一个 agent 的"分寸"，不在模型参数里，在这张门禁表里。

而这篇最想让你带走的，是那场亲手拆自己门禁的实验：

黑名单列不完所有的死法，白名单只需要定义好活着的样子。